# MathScribe: Colab runner

This notebook is a **thin runner**. It has no training or evaluation logic. It only:

1. mounts Google Drive (optional),
2. clones the repo,
3. runs `scripts/setup_env.sh --colab`,
4. logs in to the HF Hub,
5. runs the same CLI commands you run locally:
   - `TASK = "train"`: `python -m mathscribe.train.sft` (one run)
   - `TASK = "eval"`: `python -m mathscribe.eval.run_eval`, once for each model in `EVAL_RUNS` and each test set in `EVAL_TESTSETS`

Before you start:
- Select a GPU runtime (*Runtime → Change runtime type*).
- Add the secret `HF_TOKEN` in the Colab *Secrets* panel. Training needs a token with **write** access.

Every shell step runs through `sh()`. If an install step fails, the cell stops with an error,
so *Run all* does not continue with a broken install.

**Resume after a disconnect (train):** set `RESUME = True` and run all cells again.
The trainer continues from the most advanced checkpoint: the Hub `last-checkpoint/` folder, or the local `output_dir`.

**Eval loop:**
- Each run is a separate process, so the GPU memory is free again after each run.
- A failed run does not stop the loop. The summary at the end shows each run as `done`, `skipped` or `FAILED`.
- With `USE_DRIVE = True`, the results stay on Drive. After a disconnect, *Run all* again: runs with a result are skipped.
- The last cell downloads `eval_results.zip`. Its paths match the repo. In your local clone, run:
  `unzip -o eval_results.zip && uv run python -m mathscribe.eval.leaderboard`, then commit `reports/`.

In [ ]:
# ---- Run settings: edit these ----
REPO_URL = "https://github.com/Djalal-H/FineTuning.git"
BRANCH = "main"
TASK = "eval"  # "train" or "eval"
USE_DRIVE = False  # True = outputs on Google Drive (finished eval runs survive a disconnect)
# Added to every command, e.g. ["train.max_steps=30"] or ["generation.batch_size=4"].
EXTRA_OVERRIDES = []

# TASK = "train"
CONFIG = "configs/train/sft_smoke.yaml"
HUB_REPO = "djaloul/mathscribe-sft-smoke"
RESUME = False  # True = continue from the latest checkpoint

# TASK = "eval": every model in EVAL_RUNS on every test set in EVAL_TESTSETS.
# "model" (optional) replaces model.name in the config.
# "overrides" (optional) are extra --set values for this model only.
EVAL_RUNS = [
    {"config": "configs/eval/qwen35_4b_zeroshot.yaml"},
    {"config": "configs/eval/sft_smoke.yaml", "model": "djaloul/mathscribe-sft-smoke"},
    {"config": "configs/eval/qwen35_9b_zeroshot.yaml"},
]
EVAL_TESTSETS = [
    "data/testsets/latex_ocr_test.json",  # printed, 1000 rows
    "data/testsets/crohme2014.json",  # handwritten, 986 rows
]
EVAL_SUBSET = None  # the first N rows of each test set; None = all rows
SKIP_DONE = True  # skip a run if its result file already exists

In [ ]:
import os
import subprocess
import sys

# The kernel's own Python. The install and the training both use it.
PYTHON = sys.executable
os.environ["TQDM_MININTERVAL"] = "5"  # fewer progress-bar lines in the streamed log


def sh(cmd: str) -> None:
    """Run a shell command, stream its output, and raise if it fails."""
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {cmd}")


sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv")
print("Python:", PYTHON)

In [ ]:
WORKDIR = "/content/mathscribe"
OUTPUT_ROOT = "/content/outputs"
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_ROOT = "/content/drive/MyDrive/mathscribe/outputs"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

if os.path.isdir(f"{WORKDIR}/.git"):
    sh(f"git -C {WORKDIR} fetch origin {BRANCH}")
    sh(f"git -C {WORKDIR} checkout {BRANCH}")
    sh(f"git -C {WORKDIR} reset --hard origin/{BRANCH}")
else:
    sh(f"git clone --branch {BRANCH} {REPO_URL} {WORKDIR}")
os.chdir(WORKDIR)
sh("git log -1 --oneline")

In [ ]:
# Installs mathscribe[train] into PYTHON, then checks that it imports.
sh(f"PYTHON={PYTHON} bash scripts/setup_env.sh --colab")

In [ ]:
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
sh(f"{PYTHON} -c \"from huggingface_hub import whoami; print('HF user:', whoami()['name'])\"")

In [ ]:
import json
import shlex
from pathlib import Path

import yaml

RESULTS_DIR = f"{OUTPUT_ROOT}/results"  # on Drive when USE_DRIVE = True
JOBS = []  # (name, command, result file)

if TASK == "train":
    run_name = Path(CONFIG).stem
    cmd = [PYTHON, "-m", "mathscribe.train.sft", "--config", CONFIG]
    overrides = [f"train.output_dir={OUTPUT_ROOT}/{run_name}", *EXTRA_OVERRIDES]
    if HUB_REPO:
        overrides += ["hub.push=true", f"hub.repo_id={HUB_REPO}"]
    if RESUME:
        cmd.append("--resume")
    for o in overrides:
        cmd += ["--set", o]
    JOBS.append((run_name, shlex.join(cmd), f"{OUTPUT_ROOT}/{run_name}/run_info.json"))
elif TASK == "eval":
    for run in EVAL_RUNS:
        with open(run["config"]) as f:
            run_name = yaml.safe_load(f)["run_name"]
        for manifest in EVAL_TESTSETS:
            with open(manifest) as f:
                testset = json.load(f)
            rows = len(testset["rows"])
            n = rows if EVAL_SUBSET is None else min(EVAL_SUBSET, rows)
            result = f"{RESULTS_DIR}/{run_name}__{testset['name']}_n{n}.json"
            cmd = [PYTHON, "-m", "mathscribe.eval.run_eval", "--config", run["config"]]
            if EVAL_SUBSET is not None:
                cmd += ["--subset", str(EVAL_SUBSET)]
            overrides = [
                f"testset.manifest={manifest}",
                f"output.dir={OUTPUT_ROOT}/eval",
                f"output.results_dir={RESULTS_DIR}",
                f"output.leaderboard={OUTPUT_ROOT}/leaderboard.md",
                f"render.cache_dir={OUTPUT_ROOT}/render_cache",
                *EXTRA_OVERRIDES,
                *run.get("overrides", []),
            ]
            if run.get("model"):
                overrides.append(f"model.name={run['model']}")
            for o in overrides:
                cmd += ["--set", o]
            JOBS.append((f"{run_name} on {testset['name']} (n={n})", shlex.join(cmd), result))
else:
    raise ValueError(f"TASK must be 'train' or 'eval', got {TASK!r}")

for name, cmd, _ in JOBS:
    print(f"# {name}\n{cmd}\n")

In [ ]:
# This cell runs the jobs. A failed eval run does not stop the loop.
import time

STATUS = []
for i, (name, cmd, result) in enumerate(JOBS, start=1):
    print(f"\n===== [{i}/{len(JOBS)}] {name} =====", flush=True)
    if TASK == "eval" and SKIP_DONE and os.path.exists(result):
        print(f"Skipped: {result} exists.")
        STATUS.append((name, "skipped", 0.0))
        continue
    t0 = time.time()
    try:
        sh(cmd)
        STATUS.append((name, "done", time.time() - t0))
    except RuntimeError as e:
        print(f"!! {e}")
        STATUS.append((name, "FAILED", time.time() - t0))
        if TASK == "train":
            raise

print("\n===== Summary =====")
for name, status, seconds in STATUS:
    print(f"{status:8} {seconds / 60:6.1f} min  {name}")

In [ ]:
import glob
import zipfile

if TASK == "train":
    with open(JOBS[0][2]) as f:
        info = json.load(f)
    print(json.dumps(info, indent=2)[:3000])
else:
    from google.colab import files

    # One zip with the repo paths: reports/results/<run>__<testset>_n<N>.json and
    # outputs/eval/<run>/<testset>_n<N>/predictions.jsonl (to score again without a GPU).
    bundle = "/content/eval_results.zip"
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
        for result in sorted(glob.glob(f"{RESULTS_DIR}/*.json")):
            run_name, tag = Path(result).stem.split("__")
            z.write(result, f"reports/results/{Path(result).name}")
            predictions = f"{OUTPUT_ROOT}/eval/{run_name}/{tag}/predictions.jsonl"
            if os.path.exists(predictions):
                z.write(predictions, f"outputs/eval/{run_name}/{tag}/predictions.jsonl")
            with open(result) as f:
                m = json.load(f)["metrics"]
            render = m.get("render_match")
            print(
                f"{Path(result).stem:55} norm. match {100 * m['normalized_match']:5.1f}%  "
                f"render match {'-' if render is None else f'{100 * render:5.1f}%'}"
            )
    print("Download:", bundle)
    files.download(bundle)